In [1]:
import sys, logging
sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from src.utils.config import EIA_API_KEY, DEVICE, DATA_RAW, DATA_PROCESSED
from src.ingestion.eia_client import fetch_hourly_demand
from src.ingestion.opsd_client import fetch_opsd_load
from src.ingestion.weather_client import fetch_weather
from src.ingestion.preprocessing import build_features

print(f"Device : {DEVICE}")
print(f"EIA key : {'SET' if EIA_API_KEY else 'NOT SET — will use OPSD'}")
print(f"Raw dir : {DATA_RAW}")

/opt/anaconda3/envs/elec_forecast/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Device : mps
EIA key : SET
Raw dir : /Users/johntrixie/Projects/elec-forecast/data/raw


In [2]:
# If you have an EIA key:
# load_df = fetch_hourly_demand(region="US48", start="2019-01-01", end="2024-01-01")

# If no EIA key yet, use OPSD Germany (no key required):
load_df = fetch_opsd_load(country="DE", start="2019-01-01", end="2024-01-01")
print(load_df.shape)
load_df.head()

INFO | Downloading OPSD time series (this may take ~2 min, ~500MB)...
INFO | Cached OPSD data → /Users/johntrixie/Projects/elec-forecast/data/raw/opsd_time_series_60min.csv
INFO | Saved OPSD load → /Users/johntrixie/Projects/elec-forecast/data/raw/opsd_DE_2019_2024.csv


(15336, 3)


,timestamp,load_mw,region
0,2019-01-01 00:00:00,41562.0,DE
1,2019-01-01 01:00:00,40100.0,DE
2,2019-01-01 02:00:00,38883.0,DE
3,2019-01-01 03:00:00,38806.0,DE
4,2019-01-01 04:00:00,38593.0,DE


In [3]:
weather_df = fetch_weather(region="DE", start="2019-01-01", end="2024-01-01")
print(weather_df.shape)
weather_df.head()

INFO | Fetching weather for DE (52.5, 13.4): 2019-01-01 → 2024-01-01
INFO |   2019: 8760 rows
INFO |   2020: 8784 rows
INFO |   2021: 8760 rows
INFO |   2022: 8760 rows
INFO |   2023: 8760 rows
INFO |   2024: 8784 rows
INFO | Saved weather data → /Users/johntrixie/Projects/elec-forecast/data/raw/weather_DE_2019_2024.csv


(43824, 7)


,timestamp,temperature_2m,wind_speed_10m,cloud_cover,shortwave_radiation,relative_humidity_2m,region
0,2019-01-01 00:00:00,7.5,19.2,95,0.0,97,DE
1,2019-01-01 01:00:00,7.5,21.9,100,0.0,95,DE
2,2019-01-01 02:00:00,7.2,23.7,100,0.0,94,DE
3,2019-01-01 03:00:00,7.2,25.7,100,0.0,92,DE
4,2019-01-01 04:00:00,7.4,34.3,100,0.0,84,DE


In [4]:
features_df = build_features(
    load_df=load_df,
    weather_df=weather_df,
    country="DE",   # change to "US" if using EIA data
    save=True,
    filename="features_DE.parquet",
)
print(features_df.shape)
print(features_df.dtypes)
features_df.head()

INFO | Starting preprocessing pipeline...
INFO |   Hourly index: 15336 rows
INFO |   Merged weather columns: ['temperature_2m', 'wind_speed_10m', 'cloud_cover', 'shortwave_radiation', 'relative_humidity_2m']
INFO |   Final shape: (15336, 34)
INFO | Saved features → /Users/johntrixie/Projects/elec-forecast/data/processed/features_DE.parquet


(15336, 34)
timestamp                 datetime64[ns]
load_mw                          float64
region                            object
temperature_2m                   float64
wind_speed_10m                   float64
cloud_cover                        int64
shortwave_radiation              float64
relative_humidity_2m               int64
hour                               int32
dayofweek                          int32
dayofyear                          int32
month                              int32
weekofyear                         int64
quarter                            int32
year                               int32
hour_sin                         float64
hour_cos                         float64
dow_sin                          float64
dow_cos                          float64
month_sin                        float64
month_cos                        float64
dayofyear_sin                    float64
dayofyear_cos                    float64
is_weekend                         int64
is_n

,timestamp,load_mw,region,temperature_2m,wind_speed_10m,cloud_cover,shortwave_radiation,relative_humidity_2m,hour,dayofweek,...,is_night,is_peak,is_holiday,load_mw_lag_24h,load_mw_lag_48h,load_mw_lag_168h,load_mw_roll_mean_24h,load_mw_roll_std_24h,load_mw_roll_mean_168h,load_mw_roll_std_168h
0,2019-01-01 00:00:00,41562.0,DE,7.5,19.2,95,0.0,97,0,1,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-01 01:00:00,40100.0,DE,7.5,21.9,100,0.0,95,1,1,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-01 02:00:00,38883.0,DE,7.2,23.7,100,0.0,94,2,1,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-01-01 03:00:00,38806.0,DE,7.2,25.7,100,0.0,92,3,1,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-01-01 04:00:00,38593.0,DE,7.4,34.3,100,0.0,84,4,1,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import pandas as pd
df = pd.read_parquet("../data/processed/features_DE.parquet")
print("Rows           :", len(df))
print("Missing load_mw:", df["load_mw"].isna().sum())
print("Date range     :", df["timestamp"].min(), "→", df["timestamp"].max())
print("Columns        :", df.columns.tolist())

Rows           : 15336
Missing load_mw: 0
Date range     : 2019-01-01 00:00:00 → 2020-09-30 23:00:00
Columns        : ['timestamp', 'load_mw', 'region', 'temperature_2m', 'wind_speed_10m', 'cloud_cover', 'shortwave_radiation', 'relative_humidity_2m', 'hour', 'dayofweek', 'dayofyear', 'month', 'weekofyear', 'quarter', 'year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos', 'is_weekend', 'is_night', 'is_peak', 'is_holiday', 'load_mw_lag_24h', 'load_mw_lag_48h', 'load_mw_lag_168h', 'load_mw_roll_mean_24h', 'load_mw_roll_std_24h', 'load_mw_roll_mean_168h', 'load_mw_roll_std_168h']
